# The REG741_APPLN_STATUS table 

The `REG741_APPLN_STATUS` table is a reference (lookup) table introduced to support the Unitary Patent (UP) system within the PATSTAT EP Register database. Its main purpose is to provide clear text descriptions for the numeric status codes that indicate the legal status of an application during the unitary phase. This table acts as a dictionary for the `STATUS` field found in the `REG701_APPLN` table. 

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG701_APPLN, REG741_APPLN_STATUS
from sqlalchemy import func

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

## Key fields in REG741_APPLN_STATUS table

### STATUS (primary key)
The `STATUS` field indicates the current legal or procedural status of the Unitary Patent. This attribute is specific to the table `REG701_APPLN` and reflects the state of the patent under the Unitary Patent system.
*   Format: A single-digit number ranging from 1 to 9.
*   Decoding the values: The numeric values serve as codes. To understand the specific meaning of each number (e.g., whether the patent is registered, has lapsed, or is revoked), reference must be made to the separate lookup table: `REG741_APPLN_STATUS`.

### STATUS_TEXT
The `STATUS_TEXT` field provides the clear, human-readable description of each Unitary Patent status. Its role is to explain the numeric status codes used in the `REG701_APPLN` table. While the `STATUS` field stores a number (for example, 9), the `STATUS_TEXT` field shows the meaning of that number in plain language (for example, “Unitary effect registered”).

In [2]:
q = (
    db.query(
        REG741_APPLN_STATUS.status,
        REG741_APPLN_STATUS.status_text
    )
    .order_by(REG741_APPLN_STATUS.status)
    .distinct()
)

res = patstat.df(q)
res

,status,status_text
0,1,Patent revoked
1,4,Request for unitary effect withdrawn
2,5,Request for unitary effect rejected
3,6,Request for unitary effect filed
4,7,Unitary effect lapsed
5,8,Request for re-establishment of rights
6,9,Unitary effect registered


## Linking REG701_APPLN to REG741_APPLN_STATUS
By performing a join using the `STATUS` field, it is possible to enrich unitary phase data with the text-translation of the of the `STATUS_TEXT` field.

In [3]:
q = (
    db.query(
        REG701_APPLN.id,
        REG701_APPLN.status,
        REG741_APPLN_STATUS.status_text
    )
    .join(
        REG741_APPLN_STATUS,
        REG741_APPLN_STATUS.status == REG701_APPLN.status
    )
    .distinct()
    .limit(1000)
)

res = patstat.df(q)
res

,id,status,status_text
0,20797527,9,Unitary effect registered
1,15188105,9,Unitary effect registered
2,16809848,9,Unitary effect registered
3,15810951,9,Unitary effect registered
4,18382243,9,Unitary effect registered
...,...,...,...
995,19724092,9,Unitary effect registered
996,21705424,9,Unitary effect registered
997,18735030,9,Unitary effect registered
998,16170493,9,Unitary effect registered


## Differences with REG403_APPLN_STATUS
Although both tables are used to decode application status values, there are important structural and conceptual differences between them. These differences are based on the stage of the patent life cycle they describe.

1. **Scope**: `REG403_APPLN_STATUS` refers to the standard European patent (EP) procedure. It tracks an application from filing through grant, refusal, or withdrawal. `REG741_APPLN_STATUS` refers exclusively to the UP phase. It becomes relevant only after a European patent has been granted and the patent holder has requested unitary effect. 

2. **Status Code Domain**: `REG403_APPLN_STATUS` uses a broader range of codes, from 1 to 18. These codes represent complex procedural states, such as "application publication", "examination in progress" and "intention to grant the patent". `REG741_APPLN_STATUS` uses a smaller and more focused range, from 1 to 9. These codes describe events specific to the unitary procedure, such as "filing a request for unitary effect", "registration of unitary effect" and "termination of unitary effect".

3. **Linking Logic and Hierarchy**: `REG403_APPLN_STATUS` is linked to the `REG101_APPLN` table through the `STATUS` field. `REG741_APPLN_STATUS` is linked to the `REG701_APPLN` table.